# FishCheck — 어종 판별 모델 학습

**목적**: 수산시장 7종 생선(가자미/개볼락/광어/도다리/방어/부시리/우럭) 이미지 분류  
**모델**: EfficientNetB0 전이학습 (ImageNet → 어종 7-class)  
**환경**: Google Colab T4 GPU

## 실행 순서
1. 런타임 → GPU 연결 확인
2. 셀 순서대로 전체 실행 (Ctrl+F9)
3. 학습 완료 후 Section 8에서 Hugging Face Hub 업로드

## 1. 환경 설정 및 패키지 설치

In [ ]:
!pip install -q huggingface-hub seaborn

import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

## 2. Google Drive 마운트 및 데이터 경로 설정

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── 본인 Google Drive 경로로 수정하세요 ──
DATA_DIR = '/content/drive/MyDrive/FishCheck/data/raw'
# -----------------------------------------

classes = sorted(os.listdir(DATA_DIR))
print('발견된 어종 폴더:', classes)
print('클래스 수:', len(classes))

for cls in classes:
    count = len(os.listdir(os.path.join(DATA_DIR, cls)))
    print(f'  {cls}: {count}장')

## 3. 데이터 로드 및 증강

In [ ]:
IMG_SIZE = 224
BATCH_SIZE = 32

train_gen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2,
    horizontal_flip=True,
    rotation_range=15,
    brightness_range=[0.8, 1.2],
    zoom_range=0.1,
    width_shift_range=0.05,
    height_shift_range=0.05,
)

val_gen = ImageDataGenerator(
    rescale=1.0/255,
    validation_split=0.2,
)

train_ds = train_gen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42,
)

val_ds = val_gen.flow_from_directory(
    DATA_DIR,
    target_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42,
)

CLASS_NAMES = list(train_ds.class_indices.keys())
NUM_CLASSES = len(CLASS_NAMES)
print('클래스 인덱스:', train_ds.class_indices)
print('학습:', train_ds.samples, '/ 검증:', val_ds.samples)

## 4. EfficientNetB0 모델 구성 (전이학습)

In [ ]:
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
)
base_model.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)
x = layers.Dense(256, activation='relu')(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = keras.Model(inputs, outputs)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

model.summary()

## 5. Phase 1 학습 (상단 레이어만)

In [ ]:
callbacks_p1 = [
    EarlyStopping(patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.5, patience=3, verbose=1),
    ModelCheckpoint('best_phase1.h5', save_best_only=True, verbose=1),
]

history1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks_p1,
)

print(f'Phase1 최고 val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

## 6. Phase 2 학습 (Fine-tuning)

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy'],
)

callbacks_p2 = [
    EarlyStopping(patience=7, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(factor=0.3, patience=4, verbose=1),
    ModelCheckpoint('fishcheck_model.h5', save_best_only=True, verbose=1),
]

history2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=30,
    callbacks=callbacks_p2,
)

print(f'Phase2 최고 val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

## 7. 결과 시각화 (학습 곡선 + 혼동 행렬)

In [ ]:
all_acc     = history1.history['accuracy']     + history2.history['accuracy']
all_val_acc = history1.history['val_accuracy'] + history2.history['val_accuracy']
all_loss    = history1.history['loss']         + history2.history['loss']
all_val_loss= history1.history['val_loss']     + history2.history['val_loss']
split       = len(history1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
ax1.plot(all_acc, label='Train'); ax1.plot(all_val_acc, label='Val')
ax1.axvline(split, color='gray', linestyle='--', label='Fine-tune 시작')
ax1.set_title('Accuracy'); ax1.legend()

ax2.plot(all_loss, label='Train'); ax2.plot(all_val_loss, label='Val')
ax2.axvline(split, color='gray', linestyle='--'); ax2.set_title('Loss'); ax2.legend()
plt.tight_layout(); plt.savefig('training_curve.png', dpi=100); plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns

model_best = keras.models.load_model('fishcheck_model.h5')
val_ds.reset()
y_pred = np.argmax(model_best.predict(val_ds), axis=1)
y_true = val_ds.classes

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('혼동 행렬'); plt.ylabel('실제'); plt.xlabel('예측')
plt.tight_layout(); plt.savefig('confusion_matrix.png', dpi=100); plt.show()

print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

## 8. Hugging Face Hub 업로드

In [ ]:
from huggingface_hub import HfApi, login

# huggingface.co → Settings → Access Tokens에서 발급 후 입력
login()

# ── 본인 HF 사용자명으로 수정 ──
HF_USERNAME = 'your-hf-username'
# ──────────────────────────────
REPO_ID = f'{HF_USERNAME}/fishcheck-model'

api = HfApi()
api.create_repo(repo_id=REPO_ID, exist_ok=True)
api.upload_file(
    path_or_fileobj='fishcheck_model.h5',
    path_in_repo='fishcheck_model.h5',
    repo_id=REPO_ID,
)

print(f'업로드 완료! REPO_ID: {REPO_ID}')
print(f'다음 단계: CLAUDE.md와 src/model.py의 HF_REPO_ID를 [{REPO_ID}]로 업데이트하세요.')